In [39]:
import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter

### models

In [64]:
class AttentionBlock(nn.Module):
    def __init__(self, embedding_size, num_heads):
        super().__init__()
        assert embedding_size % num_heads == 0
        self.num_heads = num_heads
        self.heads_dim = embedding_size // num_heads

        self.W_q = nn.Linear(embedding_size, embedding_size)
        self.W_k = nn.Linear(embedding_size, embedding_size)
        self.W_v = nn.Linear(embedding_size, embedding_size)
        self.W_o = nn.Linear(embedding_size, embedding_size)

    def forward(self, x):
        B, L, D = x.shape

        Q = self.W_q(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)

        # no causal mask — bidirectional attention for classification
        QK_T = torch.matmul(Q, K.transpose(-1, -2)) / (self.heads_dim ** 0.5)
        attn = torch.softmax(QK_T, dim=-1)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, hidden_dim):
        super().__init__()
        self.attn = AttentionBlock(embedding_dim, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x


class PositionalEmbeddings(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, x):
        B, L = x.shape
        device = x.device  # match device to input

        position = torch.arange(L, device=device).unsqueeze(-1)
        div_term = torch.exp(
            torch.arange(0, self.embedding_dim, 2, device=device) *
            -torch.log(torch.tensor(10000.0, device=device)) / self.embedding_dim
        )

        embeddings = torch.zeros(B, L, self.embedding_dim, device=device)
        embeddings[:, :, 0::2] = torch.sin(position * div_term)
        embeddings[:, :, 1::2] = torch.cos(position * div_term)

        return embeddings


class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = (x - mean) / (var + self.epsilon).sqrt()
        return self.gamma * x + self.beta


criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3235/5081]))

class Classifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, heads_num, vocab_size, layers_num, dropout):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.positional_embeddings = PositionalEmbeddings(embedding_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, heads_num, hidden_dim)
            for _ in range(layers_num)
        ])
        self.ln_f = LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.clf = nn.Linear(embedding_dim, 1)  # single logit for BCE

    def forward(self, x, targets=None):
        x = self.embeddings(x) + self.positional_embeddings(x)
        x = self.blocks(x)
        x = self.dropout(self.ln_f(x))
        x = x[:, 0, :]             # [CLS] token
        logits = self.clf(x)       # (B, 1)

        loss = None
        if targets is not None:
            loss = criterion(logits.squeeze(-1), targets.float())

        return logits, loss

### data prep

In [ ]:
LATIN_ALPHABET = 'abcdefghijklmnopqrstuvwxyz'
RUSSIAN_ALPHABET = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
MAX_LEN = 256
MIN_LEN = 10
PATH = "reviews/"

import re
from sklearn.model_selection import train_test_split

def prepare_data(batch_size):

    def get_reviews(positive=True):
        path = PATH + ("positive" if positive else "negative") + ".txt"
        with open(path, 'r') as f:
            return f.read()

    def process(review):
        review = review.lower()
        review = ''.join([
            c for c in review
            if c in (' ', '!', '?') or c in LATIN_ALPHABET or c in RUSSIAN_ALPHABET
        ])
        review = re.sub(r'!+', '!!!', review)
        review = re.sub(r'\?+', '???', review)
        review = review.strip()
        if len(review) < MIN_LEN:
            return None
        return review

    def build_vocab(reviews):
        symbols = sorted(set(''.join(reviews)))
        # 0 = PAD, 1 = CLS
        mapping = {char: idx + 2 for idx, char in enumerate(symbols)}
        vocab_size = len(mapping) + 2
        return mapping, vocab_size

    def tokenize_review(review, mapping):
        tokens = [1] + [mapping.get(c, 0) for c in review]  # prepend CLS=1
        if len(tokens) < MAX_LEN:
            tokens = tokens + [0] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]
        return tokens

    positive_reviews = get_reviews(True)
    negative_reviews = get_reviews(False)

    pos = [process(r) for r in positive_reviews.replace('\xa0', '\n').split('\n') if r]
    neg = [process(r) for r in negative_reviews.replace('\xa0', '\n').split('\n') if r]

    pos = [r for r in pos if r is not None]
    neg = [r for r in neg if r is not None]

    mapping, vocab_size = build_vocab(pos + neg)

    pos_tokens = [tokenize_review(r, mapping) for r in pos]
    neg_tokens = [tokenize_review(r, mapping) for r in neg]

    inputs = pos_tokens + neg_tokens
    labels = [1] * len(pos_tokens) + [0] * len(neg_tokens)

    train_inputs, val_inputs, train_labels, val_labels = train_test_split(
        inputs, labels, test_size=0.2, shuffle=True, stratify=labels, random_state=42
    )

    class ReviewDataset(torch.utils.data.Dataset):
        def __init__(self, inputs, labels):
            self.inputs = inputs
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            x = torch.tensor(self.inputs[idx], dtype=torch.long)
            y = torch.tensor(self.labels[idx], dtype=torch.float)
            return x, y

    train_set = ReviewDataset(train_inputs, train_labels)
    val_set = ReviewDataset(val_inputs, val_labels)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, vocab_size

### data prep (attempt 2)

In [60]:
LATIN_ALPHABET = 'abcdefghijklmnopqrstuvwxyz'
RUSSIAN_ALPHABET = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
MAX_LEN = 256
MIN_LEN = 10
PATH = "reviews/"

import re
from sklearn.model_selection import train_test_split

def prepare_data(batch_size, win_size, win_hop):

    def get_reviews(positive=True):
        path = PATH + ("positive" if positive else "negative") + ".txt"
        with open(path, 'r') as f:
            return f.read()

    def process(review):
        review = review.lower()
        review = ''.join([
            c for c in review
            if c in (' ', '!', '?') or c in LATIN_ALPHABET or c in RUSSIAN_ALPHABET
        ])
        review = re.sub(r'!+', '!!!', review)
        review = re.sub(r'\?+', '???', review)
        review = review.strip()
        if len(review) < MIN_LEN:
            return None
        return review

    def build_vocab(reviews):
        symbols = sorted(set(''.join(reviews)))
        # 0 = PAD, 1 = CLS
        mapping = {char: idx + 2 for idx, char in enumerate(symbols)}
        vocab_size = len(mapping) + 2
        return mapping, vocab_size

    def tokenize_review(review, mapping):
        tokens = [mapping.get(c, 0) for c in review]  # no CLS
        if len(tokens) < MAX_LEN:
            tokens = tokens + [0] * (MAX_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_LEN]
        return tokens

    positive_reviews = get_reviews(True)
    negative_reviews = get_reviews(False)

    pos = [process(r) for r in positive_reviews.replace('\xa0', '\n').split('\n') if r]
    neg = [process(r) for r in negative_reviews.replace('\xa0', '\n').split('\n') if r]

    pos = [r for r in pos if r is not None]
    neg = [r for r in neg if r is not None]

    # split into many overlapping sequences
    pos = " ".join(pos)
    neg = " ".join(neg)

    split_pos = int(len(pos) * 0.8)
    split_neg = int(len(neg) * 0.8)

    pos_train = pos[:split_pos]
    pos_test = pos[split_pos:]
    neg_train = neg[:split_neg]
    neg_test = neg[split_neg:]

    def split_sequence(sequence, win_size, win_hop):
        return [sequence[i:i + win_size] for i in range(0, len(sequence) - win_size + 1, win_hop)]

    pos_train_seq = split_sequence(pos_train, win_size, win_hop)
    pos_test_seq = split_sequence(pos_test, win_size, win_hop)
    neg_train_seq = split_sequence(neg_train, win_size, win_hop)
    neg_test_seq = split_sequence(neg_test, win_size, win_hop)

    mapping, vocab_size = build_vocab(pos + neg)

    pos_train_tokens = [tokenize_review(r, mapping) for r in pos_train_seq]
    pos_test_tokens = [tokenize_review(r, mapping) for r in pos_test_seq]
    neg_train_tokens = [tokenize_review(r, mapping) for r in neg_train_seq]
    neg_test_tokens = [tokenize_review(r, mapping) for r in neg_test_seq]

    train_inputs = pos_train_tokens + neg_train_tokens
    train_labels = [1] * len(pos_train_tokens) + [0] * len(neg_train_tokens) 

    test_inputs = pos_test_tokens + neg_test_tokens
    test_labels = [1] * len(pos_test_tokens) + [0] * len(neg_test_tokens) 

    class ReviewDataset(torch.utils.data.Dataset):
        def __init__(self, inputs, labels):
            self.inputs = inputs
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            x = torch.tensor(self.inputs[idx], dtype=torch.long)
            y = torch.tensor(self.labels[idx], dtype=torch.float)
            return x, y

    train_set = ReviewDataset(train_inputs, train_labels)
    val_set = ReviewDataset(test_inputs, test_labels)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)

    print(f"pos train: {len(pos_train_seq)}, pos val: {len(pos_test_seq)}")
    print(f"neg train: {len(neg_train_seq)}, neg val: {len(neg_test_seq)}")

    return train_loader, val_loader, vocab_size

### model training


In [72]:
CONTEXT_LEN = 128
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
BATCH_SIZE = 64
NUM_HEADS = 1
LAYERS = 6
DEVICE = 'cuda' if torch.accelerator.is_available() else 'cpu'
EPOCHS = 100
LR = 1e-3
DROPOUT = 0.4
WIN_HOP = 16

In [73]:
train_loader, val_loader, vocab_size = prepare_data(BATCH_SIZE, CONTEXT_LEN, WIN_HOP)

model = Classifier(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    heads_num=NUM_HEADS,
    vocab_size=vocab_size,
    layers_num=LAYERS,
    dropout=DROPOUT,
)
writer = SummaryWriter(log_dir=f"runs/lr{LR}_dropout{DROPOUT}_layers{LAYERS}")

pos train: 10161, pos val: 2535
neg train: 6469, neg val: 1612


In [ ]:
model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1226/2228]).to(DEVICE))

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split, loader in [("train", train_loader), ("val", val_loader)]:
        losses = []
        correct = 0
        total = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits, loss = model(x, y)
            losses.append(loss.item())
            preds = (torch.sigmoid(logits.squeeze(-1)) > 0.5).float()  # (B,)
            correct += (preds == y).sum().item()
            total += y.size(0)
        out[split] = {
            "loss": sum(losses) / len(losses),
            "acc": correct / total
        }
    model.eval()
    return out

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    metrics = estimate_loss()

    # tensorboard ---------------------------------------------------
    writer.add_scalar("Loss/train", metrics["train"]["loss"], epoch)
    writer.add_scalar("Loss/val", metrics["val"]["loss"], epoch)
    writer.add_scalar("Accuracy/train", metrics["train"]["acc"], epoch)
    writer.add_scalar("Accuracy/val", metrics["val"]["acc"], epoch)
    # tensorboard ---------------------------------------------------

    print(f"epoch {epoch+1:3d} | "
          f"train loss {metrics['train']['loss']:.4f} | train acc {metrics['train']['acc']:.4f} | "
          f"val loss {metrics['val']['loss']:.4f} | val acc {metrics['val']['acc']:.4f}")
    
writer.close()